# CALPHAD workflow with ATAT and machine-learned potentials

This notebook fits a CALPHAD database for Ni-Re with the `CalphadMaker`
workflow and draws the phase diagram. The energies come from three MACE
potentials and one GRACE potential, so no DFT is needed.

Section 1 sets up the workflow for Ni-Re and draws the phase diagram for each
potential. Section 2 checks the size of the liquid cell and the scatter between
MD runs. Section 3 compares the phase diagrams with experiment.

## Background

The workflow follows the `sqs2tdb` approach of ATAT (van de Walle et al.,
Calphad 58, 70 (2017)). It has four steps:

1. `sqs2tdb` copies the special quasirandom structures (SQS) of each lattice
   from the ATAT database. At level 2 these are the pure elements and the
   compositions with 25, 50 and 75 at% of each element.
2. Each solid SQS is relaxed with the force field, including the cell.
3. Each liquid SQS is repeated three times along each lattice vector, which
   gives 864 atoms. It is melted at `melt_temperature` and then run at
   `liquid_temperature`. Both runs are NPT at zero pressure. The mean potential
   energy of the second run is the energy of the liquid.
4. `sqs2tdb` fits Redlich-Kister mixing terms to the energies of each lattice
   and writes one TDB file.

For lattices in the SGTE database, `sqs2tdb` takes the free energies of the
pure elements from SGTE (Dinsdale, Calphad 15, 317 (1991)). Only the mixing
terms come from the force field. The melting points of pure Ni and Re are
therefore the experimental ones. The mixing terms do not depend on temperature.
For the solids they come from 0 K energies. For the liquid they are the excess
energies at `liquid_temperature`.

The same workflow runs with any force field in atomate2.

## Installation

The `ase` extra is needed, plus MACE and pycalphad:

```
pip install 'atomate2[ase]' 'mace-torch>=0.3.16' pycalphad
```

GRACE needs `tensorpotential`, which brings TensorFlow. PyTorch and
TensorFlow each install their own CUDA libraries, so we ran GRACE in a separate
environment:

```
pip install 'atomate2[ase]' tensorpotential pycalphad
export TF_USE_LEGACY_KERAS=1
```

GRACE saves its models in `~/.cache/grace`. Set `GRACE_CACHE` to save them
somewhere else.

ATAT is not a Python package. The workflow needs three of its programs, which
build in a few seconds:

```
curl -sSLO https://axelvandewalle.github.io/www-avdw/atat/atat3_52.tar.gz
tar xzf atat3_52.tar.gz
make -C atat/src cellcvrt nntouch lsfit
echo "set atatdir=$PWD/atat" > ~/.atat.rc
export PATH=$PWD/atat/src:$PATH
```

`sqs2tdb` calls the other programs by name, so the ATAT `src` folder must be on
the `PATH` of the shell that starts Jupyter. The next cell checks this.

In [ ]:
import os
import shutil
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

for program in ("sqs2tdb", "cellcvrt", "nntouch", "lsfit"):
    if shutil.which(program) is None:
        raise OSError(f"{program} is not on the PATH")

## The potentials

The notebook uses four foundation models. All four are released under the
Academic Software License.

- **MACE-OMAT-0-medium.** `mace_mp(model="medium-omat-0")` downloads it once and
  caches it under `~/.cache/mace`.
- **MACE-MATPES-PBE-0** and **MACE-MATPES-r2SCAN-0.** These are MACE-OMAT-0
  fine-tuned on the MatPES data set at the PBE and r2SCAN levels. They need
  `mace-torch>=0.3.10`. Download the two files from the `mace_matpes_0` release of
  [mace-foundations](https://github.com/ACEsuit/mace-foundations/releases/tag/mace_matpes_0):
  - [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  - [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)
- **GRACE-2L-OMAT.** A two-layer GRACE model fitted on OMat24. `grace_fm` of
  `tensorpotential` downloads it once by name. It runs in `float64`. atomate2
  has no name for GRACE, so the calculator is given as a dictionary that points
  to `grace_fm`.

Set the paths to the two downloaded MACE files in the next cell. The liquid MD
needs a GPU, so MACE runs on `"cuda"`. TensorFlow uses the GPU by itself. The
mixing energies are small differences of large total energies, so we used
`float64` for all runs.

In [ ]:
MACE = {"device": "cuda", "default_dtype": "float64"}
GRACE = {
    "@module": "tensorpotential.calculator.foundation_models",
    "@callable": "grace_fm",
}

# force field and calculator settings of each potential
# Replace the two paths with the files you downloaded.
FORCE_FIELDS = {
    "OMAT-0-medium": ("MACE-MP-0", {"model": "medium-omat-0", **MACE}),
    "MATPES-PBE-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-pbe-omat-ft.model", **MACE},
    ),
    "MATPES-r2SCAN-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-r2scan-omat-ft.model", **MACE},
    ),
    "GRACE-2L-OMAT": (GRACE, {"model": "GRACE-2L-OMAT"}),
}

## One calculator for all jobs

Every force field job builds its own calculator. `run_locally` runs all of
them in this one Python process. Loading MACE for each job grows the memory by
about 150 MB per job. This cell makes the jobs share one calculator. It is only
needed when many jobs run in one process, not with a workflow manager.

In [ ]:
from ase.calculators.calculator import Calculator

import atomate2.forcefields.utils as ff_utils

_ase_calculator = ff_utils.ase_calculator
_calculators: dict[str, Calculator] = {}


def shared_ase_calculator(calculator_meta: object, **kwargs: object) -> Calculator:
    """Build each calculator once and reuse it for every job."""
    key = repr((calculator_meta, sorted(kwargs.items())))
    if key not in _calculators:
        _calculators[key] = _ase_calculator(calculator_meta, **kwargs)
    _calculators[key].reset()
    return _calculators[key]


ff_utils.ase_calculator = shared_ase_calculator

## 1. Ni-Re

## The lattices

Ni is stable in FCC_A1 and Re in HCP_A3. We fit these two lattices and the
liquid. We also fit two ordered lattices of the ATAT database, NI3SN_D019 and
NI4MO_D1A. They test whether a force field makes an ordered phase stable. Each
ordered lattice has two sublattices, so each of its terms has one `order,level`
pair per sublattice. An ordered lattice needs the stable lattice of each
element, here FCC_A1 and HCP_A3.

## The temperatures

The liquid temperatures depend on the system, so they have no defaults. Re
melts at 3458 K in SGTE. We run the liquid MD at 3600 K, so that every
composition is liquid. The melt at 4500 K makes sure that every composition
melts within the 10 ps of the first run.

In [ ]:
from atomate2.forcefields.flows.calphad import CalphadMaker

LATTICES = ["FCC_A1", "HCP_A3", "NI3SN_D019", "NI4MO_D1A", "LIQUID"]


def get_maker(model: str) -> CalphadMaker:
    """Return the CALPHAD maker for Ni-Re with one of the potentials."""
    force_field, calculator_kwargs = FORCE_FIELDS[model]
    maker = CalphadMaker.from_force_field_name(
        force_field,
        melt_temperature=4500,
        liquid_temperature=3600,
        calculator_kwargs=calculator_kwargs,
        lattices=LATTICES,
    )
    maker.terms["NI3SN_D019"] = ["1,0:1,0", "2,0:1,0"]
    maker.terms["NI4MO_D1A"] = ["1,0:1,0", "2,0:1,0"]
    return maker


flow = get_maker("OMAT-0-medium").make(["Ni", "Re"])
[job.name for job in flow.jobs]

## Running the workflow

The flow computes 29 SQS, 24 solids and 5 liquids. On one A100 GPU the solid
relaxations took a few minutes together. With MACE each liquid run took 40 to
46 minutes, and with GRACE-2L-OMAT about 22 minutes. With `run_locally` the
liquid runs run one after the other, which takes about 3.5 hours with MACE. With a workflow manager the five liquid runs can run at
the same time on five GPUs. The cell is therefore off by default.

`create_folders=True` gives each job its own folder. The fit job runs
`sqs2tdb` in its folder.

In [ ]:
from jobflow import run_locally

RUN_WORKFLOW = False
if RUN_WORKFLOW:
    responses = run_locally(flow, create_folders=True, ensure_success=True)
    doc = responses[flow.output.uuid][1].output

## Checking the calculations

The output is a `CalphadDoc`. It holds the TDB file in `doc.tdb` and the energy
of each SQS, together with three checks.

- `relaxation_strain` of a solid measures how far the relaxed cell is from the
  ideal lattice. The ATAT `checkrelax` help calls values above 0.1 too large.
- `is_force_converged` says whether the relaxation reached `fmax` within its
  steps.
- `mean_squared_displacement` of a liquid grows with the length of the run. In
  a crystal it stays well below 1 Å².

In [ ]:
import pandas as pd

checks = None
if RUN_WORKFLOW:
    checks = pd.DataFrame(
        [
            {
                "lattice": calc.lattice,
                "folder": calc.folder,
                "relaxation_strain": calc.relaxation_strain,
                "is_force_converged": calc.is_force_converged,
                "msd (A^2)": calc.mean_squared_displacement,
            }
            for calc in doc.calculations
        ]
    )
checks

In our run with MACE-OMAT-0-medium all strains are below 0.034. Two NI3SN_D019
relaxations did not reach `fmax` within the default 500 steps. We repeated all
relaxations with 2000 steps and the energies did not change. The same holds for
the unconverged relaxations of the other two MACE potentials. With
GRACE-2L-OMAT the largest strain is 0.038 and four relaxations did not reach
`fmax`. We did not repeat these with more steps. With all four potentials the
mean squared displacement of the liquids is about 40 Å² for pure Re and about
200 Å² for pure Ni, so every liquid SQS has melted.

## The phase diagrams

To draw the diagrams without running the workflow, we refit the TDB files from
the energies of our runs. The energies below are the total energies in eV of
each SQS cell. The liquid energies are per 32-atom SQS, from the first of the
three MD runs of section 2.

`fit_tdb` is the last job of the flow, and `.original` calls its function
directly. It writes the `sqs2tdb` folders into the current folder, so each fit
runs in a folder of its own.

In [ ]:
import contextlib
from pathlib import Path

from atomate2.common.jobs.calphad import fit_tdb

# total energy in eV of each SQS cell from our runs, by potential and lattice
ENERGIES = {
    "OMAT-0-medium": {
        "FCC_A1": {
            "sqs_lev=0_a_Ni=1": -5.49209853,
            "sqs_lev=0_a_Re=1": -12.33654039,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -284.71462134,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -339.22431107,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -230.33194036,
        },
        "HCP_A3": {
            "sqs_lev=0_c_Ni=1": -10.96839031,
            "sqs_lev=0_c_Re=1": -24.85183259,
            "sqs_lev=1_c_Ni=0.5,c_Re=0.5": -285.27336656,
            "sqs_lev=2_c_Ni=0.25,c_Re=0.75": -340.62461569,
            "sqs_lev=2_c_Ni=0.75,c_Re=0.25": -230.47173992,
        },
        "NI3SN_D019": {
            "sqs_lev=0_d_Ni=1,h_Re=1": -85.67987041,
            "sqs_lev=0_d_Re=1,h_Ni=1": -57.26517478,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Ni=1": -304.29669863,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Re=1": -553.98891007,
            "sqs_lev=1_d_Ni=1,h_Ni=0.5,h_Re=0.5": -385.71742144,
            "sqs_lev=1_d_Re=1,h_Ni=0.5,h_Re=0.5": -468.69714088,
            "sqs_lev=2_d_Ni=0.5,d_Re=0.5,h_Ni=0.5,h_Re=0.5": -427.23088126,
        },
        "NI4MO_D1A": {
            "sqs_lev=0_a_Ni=1,b_Re=1": -34.46826631,
            "sqs_lev=0_a_Re=1,b_Ni=1": -54.93189197,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Ni=1": -164.50251049,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Re=1": -191.5431141,
            "sqs_lev=1_a_Ni=1,b_Ni=0.5,b_Re=0.5": -123.6822928,
            "sqs_lev=1_a_Re=1,b_Ni=0.5,b_Re=0.5": -233.01581704,
            "sqs_lev=2_a_Ni=0.5,a_Re=0.5,b_Ni=0.5,b_Re=0.5": -177.54910385,
        },
        "LIQUID": {
            "sqs_lev=0_a_Ni=1": -150.28636539,
            "sqs_lev=0_a_Re=1": -367.01286624,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -258.86144029,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -312.83439496,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -204.86225928,
        },
    },
    "MATPES-PBE-0": {
        "FCC_A1": {
            "sqs_lev=0_a_Ni=1": -5.4747177,
            "sqs_lev=0_a_Re=1": -12.34326536,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -285.56191987,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -339.69707156,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -230.48022471,
        },
        "HCP_A3": {
            "sqs_lev=0_c_Ni=1": -10.92828321,
            "sqs_lev=0_c_Re=1": -24.8367194,
            "sqs_lev=1_c_Ni=0.5,c_Re=0.5": -286.04978408,
            "sqs_lev=2_c_Ni=0.25,c_Re=0.75": -340.92055073,
            "sqs_lev=2_c_Ni=0.75,c_Re=0.25": -230.82737752,
        },
        "NI3SN_D019": {
            "sqs_lev=0_d_Ni=1,h_Re=1": -85.40853431,
            "sqs_lev=0_d_Re=1,h_Ni=1": -57.72741151,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Ni=1": -304.98225441,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Re=1": -553.36727309,
            "sqs_lev=1_d_Ni=1,h_Ni=0.5,h_Re=0.5": -386.47987275,
            "sqs_lev=1_d_Re=1,h_Ni=0.5,h_Re=0.5": -468.8524664,
            "sqs_lev=2_d_Ni=0.5,d_Re=0.5,h_Ni=0.5,h_Re=0.5": -428.16417685,
        },
        "NI4MO_D1A": {
            "sqs_lev=0_a_Ni=1,b_Re=1": -34.46971654,
            "sqs_lev=0_a_Re=1,b_Ni=1": -54.93295983,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Ni=1": -164.93724543,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Re=1": -191.99555333,
            "sqs_lev=1_a_Ni=1,b_Ni=0.5,b_Re=0.5": -123.71179622,
            "sqs_lev=1_a_Re=1,b_Ni=0.5,b_Re=0.5": -233.0373081,
            "sqs_lev=2_a_Ni=0.5,a_Re=0.5,b_Ni=0.5,b_Re=0.5": -177.88365554,
        },
        "LIQUID": {
            "sqs_lev=0_a_Ni=1": -149.65379724,
            "sqs_lev=0_a_Re=1": -366.30964697,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -259.28069982,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -312.87258298,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -205.16454813,
        },
    },
    "MATPES-r2SCAN-0": {
        "FCC_A1": {
            "sqs_lev=0_a_Ni=1": -11.95708834,
            "sqs_lev=0_a_Re=1": -52.35914458,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -1029.10123433,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -1351.96897539,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -705.15189284,
        },
        "HCP_A3": {
            "sqs_lev=0_c_Ni=1": -23.90892309,
            "sqs_lev=0_c_Re=1": -104.88089442,
            "sqs_lev=1_c_Ni=0.5,c_Re=0.5": -1029.40481884,
            "sqs_lev=2_c_Ni=0.25,c_Re=0.75": -1353.11344443,
            "sqs_lev=2_c_Ni=0.75,c_Re=0.25": -705.42385073,
        },
        "NI3SN_D019": {
            "sqs_lev=0_d_Ni=1,h_Re=1": -338.76180051,
            "sqs_lev=0_d_Re=1,h_Ni=1": -176.23368313,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Ni=1": -815.53092629,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Re=1": -2273.59233061,
            "sqs_lev=1_d_Ni=1,h_Ni=0.5,h_Re=0.5": -1300.033087,
            "sqs_lev=1_d_Re=1,h_Ni=0.5,h_Re=0.5": -1786.12827701,
            "sqs_lev=2_d_Ni=0.5,d_Re=0.5,h_Ni=0.5,h_Re=0.5": -1543.45688302,
        },
        "NI4MO_D1A": {
            "sqs_lev=0_a_Ni=1,b_Re=1": -100.25704698,
            "sqs_lev=0_a_Re=1,b_Ni=1": -221.63123017,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Ni=1": -562.52691324,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Re=1": -724.08151448,
            "sqs_lev=1_a_Ni=1,b_Ni=0.5,b_Re=0.5": -319.78014422,
            "sqs_lev=1_a_Re=1,b_Ni=0.5,b_Re=0.5": -966.436665,
            "sqs_lev=2_a_Ni=0.5,a_Re=0.5,b_Ni=0.5,b_Re=0.5": -642.71716391,
        },
        "LIQUID": {
            "sqs_lev=0_a_Ni=1": -356.62049012,
            "sqs_lev=0_a_Re=1": -1646.73731791,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -1002.91199015,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -1325.438211,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -679.673366,
        },
    },
    "GRACE-2L-OMAT": {
        "FCC_A1": {
            "sqs_lev=0_a_Ni=1": -5.49377845,
            "sqs_lev=0_a_Re=1": -12.32522647,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -285.05457725,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -339.43344811,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -230.52037786,
        },
        "HCP_A3": {
            "sqs_lev=0_c_Ni=1": -10.94756268,
            "sqs_lev=0_c_Re=1": -24.77506765,
            "sqs_lev=1_c_Ni=0.5,c_Re=0.5": -285.79517832,
            "sqs_lev=2_c_Ni=0.25,c_Re=0.75": -340.78359046,
            "sqs_lev=2_c_Ni=0.75,c_Re=0.25": -230.54821098,
        },
        "NI3SN_D019": {
            "sqs_lev=0_d_Ni=1,h_Re=1": -85.91382426,
            "sqs_lev=0_d_Re=1,h_Ni=1": -57.26791724,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Ni=1": -304.29129942,
            "sqs_lev=1_d_Ni=0.5,d_Re=0.5,h_Re=1": -553.94491608,
            "sqs_lev=1_d_Ni=1,h_Ni=0.5,h_Re=0.5": -386.05044014,
            "sqs_lev=1_d_Re=1,h_Ni=0.5,h_Re=0.5": -469.31958256,
            "sqs_lev=2_d_Ni=0.5,d_Re=0.5,h_Ni=0.5,h_Re=0.5": -427.89276251,
        },
        "NI4MO_D1A": {
            "sqs_lev=0_a_Ni=1,b_Re=1": -34.56108084,
            "sqs_lev=0_a_Re=1,b_Ni=1": -54.8895829,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Ni=1": -164.70384208,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5,b_Re=1": -191.77955326,
            "sqs_lev=1_a_Ni=1,b_Ni=0.5,b_Re=0.5": -123.8971733,
            "sqs_lev=1_a_Re=1,b_Ni=0.5,b_Re=0.5": -232.79282086,
            "sqs_lev=2_a_Ni=0.5,a_Re=0.5,b_Ni=0.5,b_Re=0.5": -177.85394644,
        },
        "LIQUID": {
            "sqs_lev=0_a_Ni=1": -149.86403747,
            "sqs_lev=0_a_Re=1": -366.85456453,
            "sqs_lev=1_a_Ni=0.5,a_Re=0.5": -259.01439416,
            "sqs_lev=2_a_Ni=0.25,a_Re=0.75": -313.07869116,
            "sqs_lev=2_a_Ni=0.75,a_Re=0.25": -204.9256195,
        },
    },
}

maker = get_maker("OMAT-0-medium")
terms = {lattice: maker.terms[lattice] for lattice in LATTICES}
tdbs = {}
for model, by_lattice in ENERGIES.items():
    calculations = [
        {"lattice": lattice, "folder": folder, "energy": energy}
        for lattice, by_folder in by_lattice.items()
        for folder, energy in by_folder.items()
    ]
    fit_dir = Path("calphad_fits", model)
    fit_dir.mkdir(parents=True, exist_ok=True)
    with contextlib.chdir(fit_dir):
        doc = fit_tdb.original(["Ni", "Re"], maker.level, terms, calculations)
        tdbs[model] = doc.tdb

In [ ]:
import matplotlib.pyplot as plt
from pycalphad import Database, binplot
from pycalphad import variables as v

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
axes = axes.flatten()
for ax, (model, tdb) in zip(axes, tdbs.items(), strict=True):
    db = Database.from_string(tdb, fmt="tdb")
    binplot(
        db,
        ["NI", "RE"],
        sorted(db.phases),
        {v.X("RE"): (0, 1, 0.02), v.T: (1000, 4000, 25), v.P: 101325, v.N: 1},
        plot_kwargs={"ax": ax},
    )
    ax.set_title(model)
for ax in axes[[1, 3]]:
    ax.set_ylabel("")
for ax in axes[:2]:
    ax.set_xlabel("")
for ax in axes[[0, 2, 3]]:
    ax.get_legend().remove()
fig.tight_layout()
plt.show()

The four potentials give the same kind of diagram.

- The FCC liquidus falls from the melting point of Ni as Re is added, or stays
  almost flat with GRACE-2L-OMAT. It ends in a eutectic between 1440 and 1700 K.
- The HCP liquidus rises to the melting point of Re.
- At lower temperatures the ordered NI4MO_D1A and NI3SN_D019 phases become
  stable.

The mixing terms of the liquid and of FCC_A1 decide most of this. In J/mol:

| Potential | L0 FCC_A1 | L0 HCP_A3 | L0 LIQUID |
|---|---|---|---|
| MACE-OMAT-0-medium | +7168 | +15938 | −2892 |
| MACE-MATPES-PBE-0 | −2723 | +2481 | −16195 |
| MACE-MATPES-r2SCAN-0 | +3980 | +14100 | −14467 |
| GRACE-2L-OMAT | +1358 | +892 | −9181 |

With all four potentials, mixing lowers the energy of the liquid more than that
of FCC. Adding Re to Ni therefore stabilizes the liquid and lowers the
liquidus. With GRACE-2L-OMAT the solid mixing terms are close to zero, so the
eutectic sits just below the melting point of Ni. Section 3 compares this with
experiment.

## 2. Size of the liquid cell and scatter between runs

The liquid energy is a time average over one MD run, so it carries statistical
noise. It can also depend on the size of the cell. We ran each liquid SQS with
three random initial velocities, set by `mb_velocity_seed` of both liquid
makers. With the MACE potentials we repeated each SQS two, three and four times
along each lattice vector, which gives 256, 864 and 2048 atoms. With
GRACE-2L-OMAT we only ran 864 atoms. The solids are the same in all runs.

The loop below runs 9 workflows for each potential, so it is off by default.
Each workflow also repeats the solid relaxations, which take about 2 minutes.

In [ ]:
RUN_LIQUID_SCAN = False
if RUN_LIQUID_SCAN:
    liquid_docs = {}
    for model in FORCE_FIELDS:
        for supercell in (2, 3, 4):
            for seed in (1, 2, 3):
                maker = get_maker(model)
                maker.liquid_supercell = supercell
                maker.liquid_melt_maker.mb_velocity_seed = seed
                maker.liquid_md_maker.mb_velocity_seed = seed
                flow = maker.make(["Ni", "Re"])
                root_dir = f"runs/{model}_{supercell}_{seed}"
                os.makedirs(root_dir, exist_ok=True)
                responses = run_locally(
                    flow, create_folders=True, root_dir=root_dir, ensure_success=True
                )
                doc = responses[flow.output.uuid][1].output
                liquid_docs[model, supercell, seed] = doc

In [ ]:
import numpy as np

# L0 of LIQUID in J/mol from our runs, by potential and number of atoms, seeds 1-3
L0_LIQUID = {
    "OMAT-0-medium": {
        256: (-713, -1889, -1438),
        864: (-2892, -3776, -3475),
        2048: (-2980, -3145, -3171),
    },
    "MATPES-PBE-0": {
        256: (-18711, -15533, -18916),
        864: (-16195, -17356, -17872),
        2048: (-17794, -16912, -15894),
    },
    "MATPES-r2SCAN-0": {
        256: (-15507, -13384, -15209),
        864: (-14467, -16199, -16556),
        2048: (-15882, -15249, -14620),
    },
    "GRACE-2L-OMAT": {864: (-9181, -5745, -7033)},
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (model, by_size) in zip(axes, L0_LIQUID.items(), strict=True):
    sizes = np.array(list(by_size))
    values = np.array(list(by_size.values()))
    for i, size in enumerate(sizes):
        ax.plot([size] * 3, values[i], "o", color="gray", mfc="white")
    ax.errorbar(sizes, values.mean(axis=1), values.std(axis=1, ddof=1), fmt="o-")
    ax.set_title(model)
    ax.set_xscale("log")
    ax.set_xlim(180, 2900)
    ax.set_xticks([256, 864, 2048], ["256", "864", "2048"])
    ax.minorticks_off()
    ax.set_xlabel("Atoms in the liquid cell")
axes[0].set_ylabel("L0 of LIQUID (J/mol)")
fig.tight_layout()
plt.show()

pd.DataFrame(
    {
        model: {
            size: f"{np.mean(v):.0f} ± {np.std(v, ddof=1):.0f}"
            for size, v in by_size.items()
        }
        for model, by_size in L0_LIQUID.items()
    }
)

Open markers are the three runs, filled markers their mean with the standard
deviation.

- From 864 to 2048 atoms the mean changes by 270 to 490 J/mol. This is within
  the scatter between the runs. The liquid is converged at 864 atoms.
- From 256 to 864 atoms the mean of MACE-OMAT-0-medium changes by 2030 J/mol.
  This is about five times its statistical error. With the two MatPES potentials
  the change is within the scatter. 256 atoms is too small for at least one of
  the three potentials, so the workflow uses 864 atoms by default.
- With the MACE potentials the scatter between runs is 100 to 1100 J/mol at
  864 and 2048 atoms. With GRACE-2L-OMAT it is 1730 J/mol at 864 atoms. It
  moves the eutectic of section 1 by up to 55 K. For a converged result,
  average the energies of a few runs.
- On one A100 GPU a MACE liquid run took about 15 minutes at 256 atoms, 43
  minutes at 864 atoms and 100 minutes at 2048 atoms.

`liquid_supercell` sets the number of repeats. A force field can need a
different size than the potentials here, so check it for a new system.

## 3. Comparison with experiment

In experiment Ni-Re has a peritectic, L + HCP → FCC, at 1834 K (1561 °C). The
liquid at the peritectic has about 11 at% Re, FCC 20 at% and HCP 60 at%
(Boettinger et al., Metall. Mater. Trans. A (2019)). The FCC liquidus rises from
the melting point of Ni as Re is added. As far as we know, no ordered phase has
been reported in Ni-Re.

The table gives the lowest temperature at which the liquid is stable with each
potential and each of the three MD runs at 864 atoms. We found it with
`pycalphad.equilibrium` on a grid with steps of 0.01 in composition and 5 K in
temperature.

In [ ]:
# lowest temperature in K at which the liquid is stable, 864 atoms, seeds 1-3
LOWEST_LIQUID_T = {
    "OMAT-0-medium": (1640, 1615, 1625),
    "MATPES-PBE-0": (1605, 1575, 1565),
    "MATPES-r2SCAN-0": (1440, 1400, 1385),
    "GRACE-2L-OMAT": (1695, 1715, 1710),
}
pd.DataFrame(LOWEST_LIQUID_T, index=["seed 1", "seed 2", "seed 3"])

None of the four potentials gives the experimental diagram.

- All four give a eutectic 120 to 450 K below the experimental peritectic. The
  FCC liquidus falls or stays flat instead of rising. GRACE-2L-OMAT comes
  closest.
- All four make ordered phases stable. The highest temperature at which each
  ordered phase is stable, on a grid with steps of 0.01 in composition and 10 K
  in temperature, is:

| Potential | NI4MO_D1A | NI3SN_D019 |
|---|---|---|
| MACE-OMAT-0-medium | 1160 K | 1210 K |
| MACE-MATPES-PBE-0 | 1670 K | 1370 K |
| MACE-MATPES-r2SCAN-0 | 1450 K | 1350 K |
| GRACE-2L-OMAT | 1690 K | 1910 K |

The scatter between the MD runs is much smaller than these differences. Two
approximations of the method can contribute. The solid mixing terms are 0 K
energies, while the liquid ones come from MD at 3600 K. The excess vibrational
entropy of mixing is left out for all phases. The force fields can also simply
give the wrong mixing energies. A comparison with DFT mixing energies of the
same SQS would tell these apart.

## Known limitations

The mixing terms do not depend on temperature. The vibrational and short-range
order options of `sqs2tdb` are not used.

The liquid mixing terms come from one temperature, `liquid_temperature`. They
are used at all temperatures.